# Trabalho Prático III: Recomendador de Disciplinas com NLP e Busca Semântica

**Disciplina:** Fundamentos de Inteligência Artificial (FIA) - Engenharia de Sistemas (UFMG)

**Objetivo:** Construir um sistema (base para um ChatBot) que permite aos estudantes interagirem em linguagem natural para montagem de seu plano de estudos. O modelo deve sugerir disciplinas que se encaixem no perfil de interesse, respeitando pré-requisitos, a grade curricular e equilibrando a carga de dificuldade por semestre.

Neste notebook, abordaremos a arquitetura utilizando **Sentence-BERT (SBERT)**, ideal para gerar embeddings semânticos e realizar busca por similaridade.

---

## O Pipeline do Modelo

Para processar o perfil do aluno e sugerir as melhores disciplinas, estruturamos 4 etapas:

### 1. Curadoria de Dados
Agregamos informações do PPC e MAPA de ofertas da UFMG, cruzando com planilhas de avaliação estudantil (Média de Dificuldade, Trabalho, etc.).

### 2. Embeddings Semânticos (Sentence-BERT)
Diferente de abordagens clássicas (como TF-IDF) ou de fine-tuning custoso de todo o BERT para classificação binária, utilizamos um modelo **Sentence-BERT (`neuralmind/bert-base-portuguese-cased`)** para converter textos (ementas e históricos) em vetores densos (embeddings) em um espaço onde textos semanticamente similares ficam próximos.

### 3. Busca por Similaridade e Geração de Perfis
Utilizamos a **Similaridade de Cosseno** para encontrar a distância entre o vetor do perfil do aluno (o que ele gosta / já fez) e os vetores das disciplinas candidatas.

### 4. Aplicação de Regras (Grade e Dificuldade)
As disciplinas mais similares passam por um filtro estrutural:
* A disciplina exige pré-requisitos não cumpridos? (Se sim, é descartada).
* A soma da dificuldade das disciplinas escolhidas ultrapassa o limite aceitável do semestre? (Se sim, substituímos por uma menos densa).


In [ ]:
import pandas as pd
import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

from src.config import *
from src.data_curation import get_discipline_table, get_completed_profile_text, get_discipline_text
from src.sbert_encoder import load_sbert_model, encode_texts, encode_disciplines, semantic_search

print("Dependências carregadas com sucesso!")

### 1. Carregando e Visualizando os Dados

Carregamos nosso dataset unificado (`disciplinas_engsis.json`) contendo 80 disciplinas da grade de Engenharia de Sistemas para a oferta 2026/1.

In [ ]:
# Carregar tabela de disciplinas curadas
df_disciplinas = get_discipline_table()

print(f"Total de disciplinas: {len(df_disciplinas)}")
display(df_disciplinas.head())

### 2. Carregando o Modelo SBERT e Gerando Vetores (Embeddings)

Aqui carregamos o modelo `neuralmind/bert-base-portuguese-cased` adaptado via Sentence Transformers. Na prática, este modelo foi treinado em corpus em português e ajustado para gerar representações densas ricas de sentenças e parágrafos.

In [ ]:
# Carregar o modelo Sentence-BERT
# A primeira vez fará o download dos pesos do HuggingFace (pode demorar um pouco)
model = load_sbert_model()

# Gerar o vetor semântico (embedding) para todas as disciplinas da base de dados.
# O texto de entrada concatena o Nome, a Área e a Ementa da disciplina.
corpus_embeddings = encode_disciplines(model, df_disciplinas)

print(f"\nFormato da matriz de embeddings: {corpus_embeddings.shape}")

### 3. Simulando um Perfil de Estudante e Busca Semântica

Vamos supor um aluno que está no 3º período e gostou muito das disciplinas de Computação e Matemática que fez até agora. Vamos gerar um "perfil textual" e calcular a similaridade de cosseno com as disciplinas candidatas.

In [ ]:
# Aluno concluiu o ciclo básico e se interessa por software e matemática aplicada.
disciplinas_concluidas = ['DCC203', 'DCC204', 'MAT001', 'MAT039']

# Texto do perfil baseado no histórico
perfil_texto = get_completed_profile_text(df_disciplinas, disciplinas_concluidas)
print(f"Texto do Perfil:\n{perfil_texto}\n")

# Adicionamos um interesse em linguagem natural simulando um input de ChatBot
query_texto = perfil_texto + " Tenho muito interesse em desenvolvimento de software, estrutura de dados e programação orientada a objetos."
print(f"Input Final do Aluno (ChatBot):\n{query_texto}\n")

# Codificar a query (o perfil do aluno)
query_embedding = encode_texts(model, [query_texto], batch_size=1)[0]

# Realizar a busca semântica (top 15 resultados)
hits = semantic_search(query_embedding, corpus_embeddings, top_k=15)

print("==== TOP DISCIPLINAS POR SIMILARIDADE SEMÂNTICA ====")
for hit in hits:
    idx = hit['index']
    score = hit['score']
    disc = df_disciplinas.iloc[idx]
    print(f"[{score:.4f}] {disc['codigo']} - {disc['nome']} (Período {disc['periodo_sugerido']})")

### 4. Filtragem Estrutural (Regras de Negócio)

Embora o SBERT ache matérias super interessantes para o perfil, **nós não podemos sugerir disciplinas cujos pré-requisitos o aluno não tem, nem lotar o semestre com as matérias mais difíceis do curso ao mesmo tempo.**

Vamos aplicar o algoritmo de filtro guloso (`greedy_balance_by_difficulty`) que checa os pré-requisitos e equilibra o semestre.

In [ ]:
from src.recommender_rules import filter_recommended_candidates, greedy_balance_by_difficulty

# Construir DataFrame de candidatos baseado nos hits
candidatas_list = []
for hit in hits:
    idx = hit['index']
    row = df_disciplinas.iloc[idx].copy()
    row['predicted_probability'] = hit['score']  # Usamos o score de cosseno como probabilidade
    row['difficulty'] = row['dificuldade_geral']
    candidatas_list.append(row)

df_candidatas = pd.DataFrame(candidatas_list)

# 1. Filtrar pré-requisitos não satisfeitos (e remover as que já foram feitas)
df_validas = filter_recommended_candidates(df_candidatas, disciplinas_concluidas)
df_validas = df_validas[~df_validas['codigo'].isin(disciplinas_concluidas)]

print(f"Disciplinas aprovadas pelos pré-requisitos: {len(df_validas)}")

# 2. Equilibrar a carga de dificuldade do semestre (Budget de Dificuldade = 25.0)
# Vamos supor que queremos recomendar até 4 disciplinas.
df_recomendacao_final = greedy_balance_by_difficulty(df_validas, difficulty_budget=25.0, max_courses=4)

print("\n==== RECOMENDAÇÃO FINAL PARA O PRÓXIMO SEMESTRE ====")
for _, row in df_recomendacao_final.iterrows():
    print(f"-> {row['codigo']} - {row['nome']}")
    print(f"   Dificuldade Estimada: {row['difficulty']} | Score Semântico: {row['predicted_probability']:.4f}\n")
